# 03. GitHub Exploratory Analysis (RQ1 EDA)

3-way classification(Anime/Default/Photo) profile type by GitHub activity metric — exploratory analysis.

In [ ]:
import json
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

matplotlib.rcParams['font.family'] = 'AppleGothic'
matplotlib.rcParams['axes.unicode_minus'] = False
sns.set_theme(style='whitegrid')

PROJECT_DIR = Path('../')
DATA_DIR = PROJECT_DIR / 'data'
PROC_DIR = DATA_DIR / 'processed'
RAW_DIR = DATA_DIR / 'raw'
FIG_DIR = PROJECT_DIR / 'figures'
FIG_DIR.mkdir(exist_ok=True)

df = pd.read_csv(PROC_DIR / 'classified_3cat.csv')

with open(RAW_DIR / 'enriched_users.json') as f:
    enriched = json.load(f)

metrics_df = pd.DataFrame([
    {
        'uid': u['user_id'],
        'followers': u.get('followers', 0),
        'public_repos': u.get('public_repos', 0),
        'total_stars': u.get('repos', {}).get('total_stars_received', 0),
        'total_forks': u.get('repos', {}).get('total_forks_received', 0),
        'activity_grade': u.get('activity_grade'),
        'sampling_group': u.get('sampling_group'),
    }
    for u in enriched.values()
])
df = df.merge(metrics_df, on='uid', how='left')

contrib_path = RAW_DIR / 'contributions.json'
if contrib_path.exists():
    with open(contrib_path) as f:
        contributions = json.load(f)
    contrib_df = pd.DataFrame([
        {
            'uid': int(uid),
            'commits': c.get('commits', 0),
            'prs': c.get('prs', 0),
            'issues': c.get('issues', 0),
            'reviews': c.get('reviews', 0),
            'total_contributions': c.get('total', 0),
        }
        for uid, c in contributions.items()
    ])
    df = df.merge(contrib_df, on='uid', how='left')
    print(f'commits merged: {df["commits"].notna().sum()} / {len(df)} users')
else:
    print('contributions.json missing — run pipeline step 3 (contributions)')

print(f'Total users: {len(df)} users')

colors = {'Anime': '#ff6b6b', 'Default': '#c0c0c0', 'Photo': '#4ecdc4'}
order = ['Anime', 'Photo', 'Default']

## 1. 3-way classification Default statistics

In [ ]:
type_counts = df['profile_type'].value_counts()
type_pcts = df['profile_type'].value_counts(normalize=True) * 100

summary = pd.DataFrame({'count': type_counts, 'ratio(%)': type_pcts.round(1)})
print(summary)

fig, ax = plt.subplots(figsize=(6, 6))
ax.pie(type_counts, labels=type_counts.index, autopct='%1.1f%%',
       colors=[colors.get(t, '#999') for t in type_counts.index],
       startangle=90, textprops={'fontsize': 14})
ax.set_title('profile picture 3-way classification distribution', fontsize=16)
plt.tight_layout()
plt.savefig(FIG_DIR / '3cat_distribution.png', dpi=150)
plt.show()

## 2. PFP type by activity metric distribution

In [ ]:
metrics = ['followers', 'total_stars', 'public_repos', 'total_forks']
if 'commits' in df.columns:
    metrics += ['commits', 'total_contributions']

for m in metrics:
    print(f'\n=== {m} ===')
    desc = df.groupby('profile_type')[m].describe()
    print(desc[['count', 'mean', 'std', '50%', 'max']].rename(columns={'50%': 'median'}).round(1))

In [ ]:
ncols = 3
nrows = (len(metrics) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows))
axes = np.atleast_1d(axes).flatten()
palette = [colors[t] for t in order]

for ax, m in zip(axes, metrics):
    plot_df = df[df[m] > 0].copy()
    plot_df[f'{m}_log'] = np.log10(plot_df[m] + 1)
    sns.boxplot(data=plot_df, x='profile_type', y=f'{m}_log', order=order,
                palette=palette, ax=ax, hue='profile_type', legend=False)
    ax.set_title(f'{m} (log10)', fontsize=13)
    ax.set_xlabel('')
    ax.set_ylabel('log10(value + 1)')
for ax in axes[len(metrics):]:
    ax.axis('off')

plt.suptitle('PFP type by activity metric distribution', fontsize=16, y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR / '3cat_metrics_boxplot.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. PFP type by Activity Grade distribution

In [ ]:
ct = pd.crosstab(df['profile_type'], df['activity_grade'], normalize='index') * 100
print(ct.round(1))

fig, ax = plt.subplots(figsize=(10, 5))
ct.loc[order].plot(kind='bar', stacked=True, ax=ax, colormap='Set2')
ax.set_title('PFP type by Activity Grade ratio', fontsize=14)
ax.set_xlabel('PFP type')
ax.set_ylabel('ratio (%)')
ax.legend(title='Activity Grade', bbox_to_anchor=(1.05, 1))
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(FIG_DIR / '3cat_activity_grade.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. PFP type by primary language distribution

In [ ]:
top_langs = df['top_language'].value_counts().head(10).index.tolist()
lang_df = df[df['top_language'].isin(top_langs)].copy()

ct_lang = pd.crosstab(lang_df['top_language'], lang_df['profile_type'], normalize='index') * 100

fig, ax = plt.subplots(figsize=(12, 6))
ct_lang[order].plot(kind='barh', stacked=True, ax=ax,
                     color=[colors[t] for t in order])
ax.set_title('primary language by PFP type ratio', fontsize=14)
ax.set_xlabel('ratio (%)')
ax.set_ylabel('')
ax.legend(title='PFP type')
plt.tight_layout()
plt.savefig(FIG_DIR / '3cat_language.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. sampling group by PFP type distribution

In [ ]:
ct_group = pd.crosstab(df['sampling_group'], df['profile_type'], normalize='index') * 100
print(ct_group[order].round(1))

fig, ax = plt.subplots(figsize=(10, 5))
ct_group[order].plot(kind='bar', stacked=True, ax=ax,
                      color=[colors[t] for t in order])
ax.set_title('sampling group by PFP type ratio', fontsize=14)
ax.set_xlabel('sampling group')
ax.set_ylabel('ratio (%)')
ax.legend(title='PFP type')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig(FIG_DIR / '3cat_sampling_group.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. summary table

In [ ]:
agg_spec = {
    'count': ('uid', 'count'),
    'followers_median': ('followers', 'median'),
    'followers_mean': ('followers', 'mean'),
    'total_stars_median': ('total_stars', 'median'),
    'total_stars_mean': ('total_stars', 'mean'),
    'public_repos_median': ('public_repos', 'median'),
    'public_repos_mean': ('public_repos', 'mean'),
}
if 'commits' in df.columns:
    agg_spec.update({
        'commits_median': ('commits', 'median'),
        'commits_mean': ('commits', 'mean'),
        'total_contrib_median': ('total_contributions', 'median'),
        'total_contrib_mean': ('total_contributions', 'mean'),
    })
summary_table = df.groupby('profile_type').agg(**agg_spec).round(1)

print('=== PFP type by summary ===')
summary_table